In [5]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# CIFAR100 데이터셋을 가져옵시다.
transform = transforms.ToTensor()    # 0~255 픽셀 값을 0~1 범위의 Tensor로 변환합니다.

train_dataset = torchvision.datasets.CIFAR100(root='./data', train=True,
                                              download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR100(root='./data', train=False,
                                             download=True, transform=transform)

print("train:", len(train_dataset), "test:", len(test_dataset))

Files already downloaded and verified
Files already downloaded and verified
train: 50000 test: 10000


In [6]:
model = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3), nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3), nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32 * 6 * 6, 256), nn.ReLU(),
    nn.Linear(256, 100),    # PyTorch의 CrossEntropyLoss는 softmax를 포함하므로
)                           # 마지막 레이어에 softmax를 따로 붙이지 않습니다.

print(model)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n학습 파라미터 수: {num_params:,}")

Sequential(
  (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Flatten(start_dim=1, end_dim=-1)
  (7): Linear(in_features=1152, out_features=256, bias=True)
  (8): ReLU()
  (9): Linear(in_features=256, out_features=100, bias=True)
)

학습 파라미터 수: 325,956


In [7]:
# 첫 번째 데이터(이미지, 라벨) 추출
image, label = train_dataset[0]

# 이미지의 형태(Shape) 출력
print("이미지 Shape:", image.shape)

이미지 Shape: torch.Size([3, 32, 32])


In [8]:
# 1. 실행 장치 지정 (GPU 지원 시 CUDA, 아니면 CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. 데이터부로더 설정 (학습용 데이터를 32개씩 묶고, 매 에폭마다 무작위로 섞음)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

def train_one_epoch(model): # 1 에폭(전체 데이터 1회 학습)을 실행하는 함수 정의
    model = model.to(device) # 모델을 연산 장치(GPU/CPU)로 이동
    criterion = nn.CrossEntropyLoss() # 손실함수 설정 (100개 클래스 분류용 다중 크로스 엔트로피)
    optimizer = torch.optim.Adam(model.parameters()) # 옵티마이저 설정 (가중치를 업데이트할 Adam 알고리즘)

    model.train() # 모델을 '학습 모드'로 전환 (드롭아웃, 배치 정규화 등의 레이어 활성화)
    running_loss, correct, total = 0.0, 0, 0 # 누적 손실값, 정답 개수, 총 이미지 수 초기화

    # train_loader에서 32개씩 묶인 배치(images, labels)를 하나씩 꺼내며 반복
    for step, (images, labels) in enumerate(train_loader):
        # 이미지와 정답 레이블을 모델과 동일한 연산 장치(GPU/CPU)로 이동
        images, labels = images.to(device), labels.to(device)

        # ----- [핵심 학습 5단계] -----
        optimizer.zero_grad() # 1) 이전 배치에서 구해둔 기울기(Gradient) 초기화
        outputs = model(images) # 2) 순전파(Forward): 모델에 이미지를 넣어 예측값(Output) 출력
        loss = criterion(outputs, labels) # 3) 손실 계산: 모델 예측값과 실제 정답의 오차 측정
        loss.backward() # 4) 역전파(Backward): 각 가중치(Parameter)별 기울기 계산
        optimizer.step() # 5) 가중치 업데이트: 계산된 기울기를 바탕으로 가중치를 수정

        # ----- [지표 집계] -----
        running_loss += loss.item() # 현재 배치의 손실(Loss) 값을 누적 합산
        correct += (outputs.argmax(dim=1) == labels).sum().item() # 가장 높은 확률의 예측값과 정답이 일치하는 개수 누적
        total += labels.size(0) # 처리한 전체 이미지 수 누적 (배치 크기인 32씩 증가)

        # 300번째 배치(이미지 약 9,600장)마다 중간 경과 출력
        if (step + 1) % 300 == 0:
            print(f"[{step + 1}/{len(train_loader)}] "
                  f"loss: {running_loss / (step + 1):.4f} - accuracy: {correct / total:.4f}")

# 함수 실행 (모델 1 에폭 학습 시작)
train_one_epoch(model)

[300/1563] loss: 4.3549 - accuracy: 0.0384
[600/1563] loss: 4.1444 - accuracy: 0.0701
[900/1563] loss: 3.9993 - accuracy: 0.0910
[1200/1563] loss: 3.8940 - accuracy: 0.1064
[1500/1563] loss: 3.8095 - accuracy: 0.1199


In [9]:
# 첫 번째 블록(예시)
block1 = nn.Sequential(
    nn.Conv2d(3, 64, kernel_size=3, padding=1),     # block1_conv1
    nn.ReLU(inplace=True),
    nn.Conv2d(64, 64, kernel_size=3, padding=1),    # block1_conv2
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=2, stride=2),          # block1_pool
)

print('첫 번째 블록 OK!!')

첫 번째 블록 OK!!


In [10]:
# 두 번째 블록
block2 = nn.Sequential(
    nn.Conv2d(64, 128, kernel_size=3, padding=1),     
    nn.ReLU(inplace=True),
    nn.Conv2d(128, 128, kernel_size=3, padding=1),    
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=2, stride=2),          
)

print('두 번째 블록 OK!!')

두 번째 블록 OK!!


In [11]:
# 세 번째 블록
block3 = nn.Sequential(
    nn.Conv2d(128, 256, kernel_size=3, padding=1),  
    nn.ReLU(inplace=True),
    nn.Conv2d(256, 256, kernel_size=3, padding=1),  
    nn.ReLU(inplace=True),
    nn.Conv2d(256, 256, kernel_size=3, padding=1),  
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=2, stride=2),          
)

print('세 번째 블록 OK!!')

세 번째 블록 OK!!


In [12]:
# 네 번째 블록
block4 = nn.Sequential(
    nn.Conv2d(256, 512, kernel_size=3, padding=1),  
    nn.ReLU(inplace=True),
    nn.Conv2d(512, 512, kernel_size=3, padding=1),  
    nn.ReLU(inplace=True),
    nn.Conv2d(512, 512, kernel_size=3, padding=1),  
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=2, stride=2),          
)
print('네 번째 블록 OK!!')

네 번째 블록 OK!!


In [13]:
# 다섯 번째 블록
block5 = nn.Sequential(
    nn.Conv2d(512, 512, kernel_size=3, padding=1),  
    nn.ReLU(inplace=True),
    nn.Conv2d(512, 512, kernel_size=3, padding=1),  
    nn.ReLU(inplace=True),
    nn.Conv2d(512, 512, kernel_size=3, padding=1),  
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=2, stride=2),          
)

print('다섯 번째 블록 OK!!')

다섯 번째 블록 OK!!


In [14]:
# 여섯 번째 블록 (완전 연결 계층)
# [torchvision VGG16 코드 구현] 링크의 self.classifier 부분을 유심히 보세요.
# 단, CIFAR100은 32x32 입력이라 다섯 번째 블록을 지나면 feature map이 (512, 1, 1)이 됩니다.
# 따라서 첫 Linear의 입력 차원은 512 * 7 * 7이 아니라 512가 되어야 합니다.
block6 = nn.Sequential(
    nn.Linear(512, 4096),            # 입력 512 -> 4096
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.5),               # 과적합 방지용 드롭아웃
    nn.Linear(4096, 4096),           # 4096 -> 4096
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.5),
)

print('여섯 번째 블록 OK!!')

classes = 100
classifier_out = nn.Linear(4096, classes)    # CIFAR100을 위한 모델 Output

여섯 번째 블록 OK!!


In [15]:
model = nn.Sequential(
    block1,
    block2,
    block3,
    block4,
    block5,
    nn.Flatten(),
    block6,
    classifier_out,
)

print(model)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nVGG-16 학습 파라미터 수: {num_params:,}")

Sequential(
  (0): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (1): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (2): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_

In [16]:
# 모델 학습!!
train_one_epoch(model)    # 1 Epoch만 학습합니다.

[300/1563] loss: 4.6087 - accuracy: 0.0093
[600/1563] loss: 4.6081 - accuracy: 0.0090
[900/1563] loss: 4.6077 - accuracy: 0.0092
[1200/1563] loss: 4.6074 - accuracy: 0.0090
[1500/1563] loss: 4.6072 - accuracy: 0.0092


In [17]:
# ResNet 구현에 필요한 준비를 먼저 해줍니다.

# PyTorch에서는 optimizer의 weight_decay 인자로 L2 규제를 한 번에 적용합니다.
# ex) torch.optim.SGD(model.parameters(), lr=0.1, weight_decay=1e-4)

# PyTorch BatchNorm의 momentum은 현재 배치 데이터의 반영 비율을 의미합니다. (기본값: 0.1)
# 통계치 업데이트 원리: (1 - momentum) * 과거치 + momentum * 현재치
# momentum = 0.1은 과거 기록을 90% 유지하고, 새로운 배치를 10%만 반영하겠다는 의미입니다.
BATCH_NORM_MOMENTUM = 0.1
BATCH_NORM_EPSILON = 1e-5

print('Resnet50 GoGo!!')

Resnet50 GoGo!!


In [18]:
# Q. ConvBlock 클래스를 완성합니다.
# shortcut 경로에 1x1 conv가 있어 입출력의 형태(shape)가 달라지는 블록입니다.
# (torchvision Bottleneck에서 downsample이 있는 경우에 해당합니다.)
class ConvBlock(nn.Module):
    def __init__(self, in_channels, filters, stride=2):
        super().__init__()
        filters1, filters2, filters3 = filters

        # 메인 경로를 하나로 묶기
        self.main_path = nn.Sequential(
            nn.Conv2d(in_channels, filters1, kernel_size=1, bias=False),
            nn.BatchNorm2d(filters1, momentum=BATCH_NORM_MOMENTUM, eps=BATCH_NORM_EPSILON),
            nn.ReLU(inplace=True),
            nn.Conv2d(filters1, filters2, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(filters2, momentum=BATCH_NORM_MOMENTUM, eps=BATCH_NORM_EPSILON),
            nn.ReLU(inplace=True),
            nn.Conv2d(filters2, filters3, kernel_size=1, bias=False),
            nn.BatchNorm2d(filters3, momentum=BATCH_NORM_MOMENTUM, eps=BATCH_NORM_EPSILON)
        )

        # 숏컷 경로
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels, filters3, kernel_size=1, stride=stride, bias=False),
            nn.BatchNorm2d(filters3, momentum=BATCH_NORM_MOMENTUM, eps=BATCH_NORM_EPSILON)
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # 한 줄로 계산 후 더하기
        out = self.main_path(x) + self.shortcut(x)
        return self.relu(out)

In [19]:
# Q. IdentityBlock 클래스를 완성합니다.
# shortcut이 입력을 그대로 더해 입출력의 형태(shape)가 유지되는 블록입니다.
# (torchvision Bottleneck에서 downsample이 없는 경우에 해당합니다.)
class IdentityBlock(nn.Module):
    def __init__(self, in_channels, filters):
        super().__init__()
        filters1, filters2, filters3 = filters

        # 메인 경로 (Main Path)
        self.main_path = nn.Sequential(
            nn.Conv2d(in_channels, filters1, kernel_size=1, bias=False),
            nn.BatchNorm2d(filters1, momentum=BATCH_NORM_MOMENTUM, eps=BATCH_NORM_EPSILON),
            nn.ReLU(inplace=True),
            nn.Conv2d(filters1, filters2, kernel_size=3, padding=1, bias=False),  # stride는 기본값 1
            nn.BatchNorm2d(filters2, momentum=BATCH_NORM_MOMENTUM, eps=BATCH_NORM_EPSILON),
            nn.ReLU(inplace=True),
            nn.Conv2d(filters2, filters3, kernel_size=1, bias=False),
            nn.BatchNorm2d(filters3, momentum=BATCH_NORM_MOMENTUM, eps=BATCH_NORM_EPSILON)
        )
        
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # 숏컷 경로: 변환 없이 x 그대로 더함
        out = self.main_path(x) + x
        return self.relu(out)

In [20]:
# Q. resnet50 함수를 완성합니다.
def resnet50(num_classes):
    model = nn.Sequential(
        # 1. Stem (초기 특징 추출층)
        nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
        nn.BatchNorm2d(64, momentum=BATCH_NORM_MOMENTUM, eps=BATCH_NORM_EPSILON),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=3, stride=2, padding=1),

        # 2. Stage 1 (3개 블록: ConvBlock 1개 + IdentityBlock 2개)
        ConvBlock(in_channels=64, filters=[64, 64, 256], stride=1),
        IdentityBlock(in_channels=256, filters=[64, 64, 256]),
        IdentityBlock(in_channels=256, filters=[64, 64, 256]),

        # 3. Stage 2 (4개 블록: ConvBlock 1개 + IdentityBlock 3개)
        ConvBlock(in_channels=256, filters=[128, 128, 512], stride=2),
        IdentityBlock(in_channels=512, filters=[128, 128, 512]),
        IdentityBlock(in_channels=512, filters=[128, 128, 512]),
        IdentityBlock(in_channels=512, filters=[128, 128, 512]),

        # 4. Stage 3 (6개 블록: ConvBlock 1개 + IdentityBlock 5개)
        ConvBlock(in_channels=512, filters=[256, 256, 1024], stride=2),
        IdentityBlock(in_channels=1024, filters=[256, 256, 1024]),
        IdentityBlock(in_channels=1024, filters=[256, 256, 1024]),
        IdentityBlock(in_channels=1024, filters=[256, 256, 1024]),
        IdentityBlock(in_channels=1024, filters=[256, 256, 1024]),
        IdentityBlock(in_channels=1024, filters=[256, 256, 1024]),

        # 5. Stage 4 (3개 블록: ConvBlock 1개 + IdentityBlock 2개)
        ConvBlock(in_channels=1024, filters=[512, 512, 2048], stride=2),
        IdentityBlock(in_channels=2048, filters=[512, 512, 2048]),
        IdentityBlock(in_channels=2048, filters=[512, 512, 2048]),

        # 6. Classifier (최종 분류층)
        nn.AdaptiveAvgPool2d((1, 1)),
        nn.Flatten(),
        nn.Linear(2048, num_classes)
    )
    
    return model

In [21]:
model = resnet50(num_classes=100)

print(model)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nResNet-50 학습 파라미터 수: {num_params:,}")

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): ConvBlock(
    (main_path): Sequential(
      (0): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
      (6): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (7): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (shortcut): Sequential(
      (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 